# Santa 2026 — The Elf Calculator

**Strategy:** Pool-aware binary construction (shared-prefix) + simulated annealing post-pass.

**How to use:**
1. Run **Cell 1** (writes `submit.py` to the working directory)
2. Run **Cell 2** (env probe — confirm CPUs and input files)
3. Run **Cell 3** (solves all 1,000 puzzles → `submission.csv`)
4. Run **Cell 4** (sanity check)
5. Submit `submission.csv` via *Submit* button.

> **SA budget:** `SA_TIME = 30.0` s/puzzle. Kaggle allows up to ~9 hrs total. Increase to 60+ for better scores.

In [ ]:
%%writefile submit.py
#!/usr/bin/env python3
"""
submit.py -- Santa 2026 Elf Calculator -- Self-contained solver
Generated by build_submit.py. Do not edit this file; edit src/ instead.

Usage:
    python submit.py --puzzles data/puzzles.csv --out submissions/sub.csv --cores 4 --sa-time 2.0 --verify
"""
# Standard library imports (consolidated from all modules)
from __future__ import annotations
import argparse
import csv
import json
import math
import multiprocessing
import random
import time
from collections import deque
from dataclasses import dataclass, field
from enum import Enum, auto
from typing import NamedTuple, Optional


# ===========================================================================
# MODULE: src.utils.py
# ===========================================================================
"""
src/utils.py
Core data structures, types, puzzle loading, and arithmetic helpers.
Shared by all solver modules and the verifier.
"""
import csv
import json
from dataclasses import dataclass, field
from typing import NamedTuple, Optional

# ---------------------------------------------------------------------------
# Data structures
# ---------------------------------------------------------------------------

class Instruction(NamedTuple):
    """One register-machine instruction: result = reg[src1] OP reg[src2] → reg[dst]"""
    op: str    # one of '+', '-', '*', '/'
    src1: int  # register index [0, 47]
    src2: int  # register index [0, 47]
    dst: int   # register index [0, 47]

    def to_list(self) -> list:
        return [self.op, self.src1, self.src2, self.dst]


class ValueOp(NamedTuple):
    """Value-level operation (register-index-free), used internally during search."""
    op: str
    a: int   # operand value (must be in pool at time of execution)
    b: int   # operand value (must be in pool at time of execution)
    result: int  # = apply_op(op, a, b)


@dataclass
class Puzzle:
    id: str
    seeds: list[int]
    targets: list[int]
    value_limit: int
    registers: int = 48
    # practice-only metadata
    family: Optional[str] = None
    design_tier: Optional[str] = None
    lower_bound: Optional[int] = None

    @property
    def seed_set(self) -> frozenset[int]:
        return frozenset(self.seeds)

    @property
    def target_set(self) -> frozenset[int]:
        return frozenset(self.targets)


# ---------------------------------------------------------------------------
# Puzzle loading
# ---------------------------------------------------------------------------

def load_puzzles(path: str) -> list[Puzzle]:
    """Load puzzles.csv → list[Puzzle]."""
    puzzles = []
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            puzzles.append(Puzzle(
                id=row["id"],
                seeds=json.loads(row["starting_values"]),
                targets=json.loads(row["targets"]),
                value_limit=int(row["value_limit"]),
                registers=int(row.get("registers", 48)),
            ))
    return puzzles


def load_practice_puzzles(path: str) -> list[Puzzle]:
    """Load practice_puzzles.csv → list[Puzzle] with family/tier/lb metadata."""
    puzzles = []
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            lb = row.get("lower_bound", "")
            puzzles.append(Puzzle(
                id=row["id"],
                seeds=json.loads(row["starting_values"]),
                targets=json.loads(row["targets"]),
                value_limit=int(row["value_limit"]),
                registers=int(row.get("registers", 48)),
                family=row.get("family"),
                design_tier=row.get("design_tier"),
                lower_bound=int(lb) if lb else None,
            ))
    return puzzles


# ---------------------------------------------------------------------------
# Arithmetic helpers
# ---------------------------------------------------------------------------

VALID_OPS = ("+", "-", "*", "/")


def apply_op(op: str, a: int, b: int, value_limit: int = 10**19) -> Optional[int]:
    """
    Apply op to (a, b). Returns the integer result if valid under competition
    rules, else None.
    Rules:
      - Result must be a positive integer (> 0).
      - Subtraction: a > b required (strict).
      - Division: exact (a % b == 0), b != 0.
      - Result <= value_limit.
    """
    if op == "+":
        r = a + b
    elif op == "-":
        if a <= b:
            return None
        r = a - b
    elif op == "*":
        r = a * b
    elif op == "/":
        if b == 0 or a % b != 0:
            return None
        r = a // b
    else:
        return None

    if r <= 0 or r > value_limit:
        return None
    return r


def build_one_op_map(pool_vals: set[int], value_limit: int) -> dict[int, ValueOp]:
    """
    Enumerate all values reachable in exactly one operation from pool × pool.
    Returns {result_value: ValueOp} (first path found per result value).
    Does NOT include values already in pool_vals.
    """
    F: dict[int, ValueOp] = {}
    pool_list = sorted(pool_vals)
    for a in pool_list:
        for b in pool_list:
            for op in VALID_OPS:
                r = apply_op(op, a, b, value_limit)
                if r is not None and r not in pool_vals and r not in F:
                    F[r] = ValueOp(op=op, a=a, b=b, result=r)
    return F


# ---------------------------------------------------------------------------
# Program serialisation helpers
# ---------------------------------------------------------------------------

def program_to_json(program: list[Instruction]) -> str:
    """Convert list[Instruction] to the competition's JSON string format."""
    return json.dumps([inst.to_list() for inst in program])


def program_from_json(s: str) -> list[Instruction]:
    """Parse a JSON program string back to list[Instruction]."""
    return [Instruction(op=row[0], src1=row[1], src2=row[2], dst=row[3])
            for row in json.loads(s)]


def write_submission(results: dict[str, list[Instruction]], out_path: str) -> None:
    """Write a results dict {puzzle_id: program} to a valid submission CSV."""
    import csv as _csv
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        w = _csv.writer(f)
        w.writerow(["id", "program"])
        for pid in sorted(results.keys()):
            w.writerow([pid, program_to_json(results[pid])])

# ===========================================================================
# MODULE: src.verifier.py
# ===========================================================================
"""
src/verifier.py
Competition-spec simulator and verifier. This is the oracle — every candidate
program must pass through here before being accepted or submitted.

Usage:
# [bundled]     from src.verifier import verify_program, replay_submission
"""
import csv
import json
from dataclasses import dataclass
from typing import Optional
# [bundled] from src.utils import Puzzle, Instruction, apply_op, program_from_json


# ---------------------------------------------------------------------------
# Result type
# ---------------------------------------------------------------------------

@dataclass
class VerifyResult:
    valid: bool
    instruction_count: int
    targets_hit: set[int]
    error: Optional[str] = None

    def __bool__(self) -> bool:
        return self.valid


# ---------------------------------------------------------------------------
# Single-program verifier
# ---------------------------------------------------------------------------

def verify_program(puzzle: Puzzle, program: list[Instruction]) -> VerifyResult:
    """
    Replay `program` on `puzzle` and check all competition rules.

    Rules checked (each produces a distinct error string):
      1. Register index out of range [0, N_REGS-1].
      2. Reading an uninitialized register.
      3. Subtraction result ≤ 0 (a must be strictly > b).
      4. Division: b == 0, or a % b != 0.
      5. Any result > value_limit.
      6. Any result ≤ 0 (catches add/mul edge cases; shouldn't happen with valid inputs
         but we guard anyway).
      7. After all instructions, at least one target was never produced.

    Returns a VerifyResult. `.valid` is True iff all rules pass and all targets hit.
    """
    N_REGS = puzzle.registers
    regs: list[Optional[int]] = [None] * N_REGS
    for i, v in enumerate(puzzle.seeds):
        regs[i] = v

    targets_remaining = set(puzzle.targets)
    # Seed values that happen to be targets are already hit
    for s in puzzle.seeds:
        targets_remaining.discard(s)

    for step, inst in enumerate(program):
        op, s1, s2, dst = inst.op, inst.src1, inst.src2, inst.dst

        # --- Rule 1: register indices in range ---
        for name, idx in (("src1", s1), ("src2", s2), ("dst", dst)):
            if not (0 <= idx < N_REGS):
                return VerifyResult(
                    valid=False, instruction_count=step,
                    targets_hit=set(puzzle.targets) - targets_remaining,
                    error=f"Step {step}: {name} register {idx} out of range [0,{N_REGS-1}]"
                )

        # --- Rule 2: uninitialized reads ---
        if regs[s1] is None:
            return VerifyResult(
                valid=False, instruction_count=step,
                targets_hit=set(puzzle.targets) - targets_remaining,
                error=f"Step {step}: src1 register r{s1} is uninitialized"
            )
        if regs[s2] is None:
            return VerifyResult(
                valid=False, instruction_count=step,
                targets_hit=set(puzzle.targets) - targets_remaining,
                error=f"Step {step}: src2 register r{s2} is uninitialized"
            )

        a, b = regs[s1], regs[s2]

        # --- Rules 3-6: arithmetic validity ---
        result = apply_op(op, a, b, puzzle.value_limit)
        if result is None:
            # Distinguish reason
            if op == "-" and a <= b:
                reason = f"subtraction not strictly positive: {a} - {b} = {a - b}"
            elif op == "/" and b == 0:
                reason = "division by zero"
            elif op == "/" and a % b != 0:
                reason = f"inexact division: {a} % {b} = {a % b}"
            else:
                r_raw = a + b if op == "+" else a * b
                if r_raw > puzzle.value_limit:
                    reason = f"result {r_raw} exceeds value_limit {puzzle.value_limit}"
                else:
                    reason = f"result ≤ 0"
            return VerifyResult(
                valid=False, instruction_count=step,
                targets_hit=set(puzzle.targets) - targets_remaining,
                error=f"Step {step} [{op} r{s1}={a} r{s2}={b}]: {reason}"
            )

        # Write result (read-before-write semantics already handled above)
        regs[dst] = result
        targets_remaining.discard(result)

    # --- Rule 7: all targets produced ---
    if targets_remaining:
        missed = sorted(targets_remaining)[:5]
        return VerifyResult(
            valid=False, instruction_count=len(program),
            targets_hit=set(puzzle.targets) - targets_remaining,
            error=f"Missing {len(targets_remaining)} targets: {missed}{'...' if len(targets_remaining)>5 else ''}"
        )

    return VerifyResult(
        valid=True, instruction_count=len(program),
        targets_hit=set(puzzle.targets)
    )


# ---------------------------------------------------------------------------
# Bulk verifier (full submission)
# ---------------------------------------------------------------------------

def replay_submission(
    puzzles_path: str,
    submission_path: str,
    *,
    verbose: bool = False,
) -> int:
    """
    Verify every row in submission_path against puzzles_path.
    Raises RuntimeError on the first invalid program.
    Returns total instruction count.
    """
# [bundled]     from src.utils import load_puzzles
    puzzles = {p.id: p for p in load_puzzles(puzzles_path)}
    total = 0
    seen = set()

    with open(submission_path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            pid = row["id"]
            if pid not in puzzles:
                raise RuntimeError(f"Unknown puzzle id '{pid}' in submission")
            if pid in seen:
                raise RuntimeError(f"Duplicate puzzle id '{pid}' in submission")
            seen.add(pid)

            program = program_from_json(row["program"])
            result = verify_program(puzzles[pid], program)
            if not result.valid:
                raise RuntimeError(f"Puzzle '{pid}' INVALID: {result.error}")
            total += result.instruction_count
            if verbose:
                print(f"  {pid}: {result.instruction_count} instructions — OK")

    missing = set(puzzles.keys()) - seen
    if missing:
        raise RuntimeError(f"Submission missing {len(missing)} puzzles: {sorted(missing)[:5]}")

    return total


# ---------------------------------------------------------------------------
# Convenience: verify one puzzle quickly (returns bool + count)
# ---------------------------------------------------------------------------

def quick_verify(puzzle: Puzzle, program: list[Instruction]) -> tuple[bool, int, str]:
    """Returns (valid, instruction_count, error_msg)."""
    r = verify_program(puzzle, program)
    return r.valid, r.instruction_count, r.error or ""

# ===========================================================================
# MODULE: src.classifier.py
# ===========================================================================
"""
src/classifier.py
Lightweight puzzle classifier. Labels each puzzle with a strategy hint so the
orchestrator can route it to the cheapest solver module.

The classifier runs in <1ms per puzzle. All thresholds were chosen from the
measured data in docs/06_exploratory_data_analysis.md.
"""
import math
from enum import Enum, auto
# [bundled] from src.utils import Puzzle


class PuzzleClass(Enum):
    TINY        = auto()   # value_limit < 100; solvable by exact BFS
    SMOOTH      = auto()   # ≥35% of targets are factor-smooth (lpf ≤ 2000)
    CLUSTERED   = auto()   # ≥5% of adjacent sorted target pairs within 0.5%
    GENERIC     = auto()   # everything else → meet-in-the-middle


def _smoothness_fraction(targets: list[int], lpf_threshold: int = 2000) -> float:
    """Fraction of targets whose largest prime factor is ≤ lpf_threshold."""
    if not targets:
        return 0.0
    smooth = sum(1 for t in targets if _largest_prime_factor(t) <= lpf_threshold)
    return smooth / len(targets)


def _largest_prime_factor(n: int) -> int:
    """Return the largest prime factor of n (n >= 1). Returns 1 for n==1."""
    if n <= 1:
        return 1
    lpf = 1
    for p in range(2, min(2001, int(math.isqrt(n)) + 1)):
        if n % p == 0:
            lpf = p
            while n % p == 0:
                n //= p
    if n > 1:
        lpf = n  # n itself is a prime > 2000
    return lpf


def _cluster_fraction(targets: list[int], rel_tol: float = 0.005) -> float:
    """Fraction of adjacent sorted target pairs within rel_tol of each other."""
    if len(targets) < 2:
        return 0.0
    t = sorted(targets)
    close = sum(
        1 for i in range(len(t) - 1)
        if t[i] > 0 and abs(t[i + 1] - t[i]) / t[i] <= rel_tol
    )
    return close / (len(t) - 1)


def classify(puzzle: Puzzle) -> PuzzleClass:
    """Return the PuzzleClass label for a puzzle."""
    # TINY: tiny value_limit — small state space, BFS is optimal
    if puzzle.value_limit < 100:
        return PuzzleClass.TINY

    # SMOOTH: dominant factor structure
    if _smoothness_fraction(puzzle.targets) >= 0.35:
        return PuzzleClass.SMOOTH

    # CLUSTERED: many nearby target pairs
    if _cluster_fraction(puzzle.targets) >= 0.05:
        return PuzzleClass.CLUSTERED

    return PuzzleClass.GENERIC


def classify_all(puzzles: list[Puzzle]) -> dict[str, PuzzleClass]:
    """Classify a list of puzzles. Returns {puzzle_id: PuzzleClass}."""
    return {p.id: classify(p) for p in puzzles}

# ===========================================================================
# MODULE: src.solvers.fallback.py
# ===========================================================================
"""
src/solvers/fallback.py
Fallback binary-construction solver.

Strategy: produce `1` via (seed/seed), then build each target by binary
decomposition (repeated doubling + adding 1). Never produces any value
above the target, so value_limit is automatically satisfied.

This matches the official sample_submission.csv strategy. It guarantees
a valid submission for every puzzle. All other solvers try to beat this.
"""
# [bundled] from src.utils import Puzzle, Instruction, apply_op


def fallback_solve(puzzle: Puzzle) -> list[Instruction]:
    """
    Build a valid program for `puzzle` using binary construction.
    Always succeeds. Expected cost: ~sum(floor(log2(t)) + popcount(t)) per target.
    """
    instructions: list[Instruction] = []
    seeds = puzzle.seeds
    n_seeds = len(seeds)
    value_limit = puzzle.value_limit

    # Register layout:
    #   r0..r(n_seeds-1)  → seeds (permanent)
    #   r(n_seeds)        → constant 1 (produced once, permanent)
    #   r(n_seeds+1)      → working scratch register (reused per target)

    reg_one = n_seeds       # holds the constant 1
    reg_scratch = n_seeds + 1  # working scratch

    # --- Produce constant 1 (seed0 / seed0 → reg_one) ---
    instructions.append(Instruction(op="/", src1=0, src2=0, dst=reg_one))

    # --- For each target, build it in reg_scratch by binary decomposition ---
    for target in puzzle.targets:
        # Skip targets already satisfied by a seed value
        if target in seeds:
            continue

        _build_target(target, reg_one, reg_scratch, instructions, value_limit)

    return instructions


def _build_target(
    target: int,
    reg_one: int,
    reg_scratch: int,
    instructions: list[Instruction],
    value_limit: int,
) -> None:
    """
    Append instructions to build `target` in reg_scratch using binary
    decomposition. Uses reg_one (holds 1) and reg_scratch (working buffer).

    Algorithm: set reg_scratch = 1, then process bits of target from MSB-1
    down to LSB:
      - double: reg_scratch = reg_scratch + reg_scratch
      - if bit is 1: reg_scratch = reg_scratch + reg_one
    """
    if target == 1:
        # reg_one already holds 1; copy it to scratch
        instructions.append(Instruction(op="+", src1=reg_one, src2=reg_one, dst=reg_scratch))
        # reg_scratch = 2 … that's wrong; we need 1.
        # Actually: 1 is already in reg_one; if the target is literally 1 and
        # we need it in scratch, we just use reg_one — but reg_one IS scratch
        # here if reg_one == reg_scratch. For the calling context, reg_one is
        # a separate permanent register, so just re-use reg_one conceptually:
        # the verifier counts "produces target" anywhere, so reg_one already
        # satisfies this target. We don't need an extra instruction.
        return

    # Initialise scratch to 1
    instructions.append(Instruction(op="+", src1=reg_one, src2=reg_one, dst=reg_scratch))
    # Now reg_scratch = 2. We've consumed the implicit "set to 1" by doubling.
    # So we represent: current_value = 1, then immediately double to 2.
    # This means we start processing from the second-most-significant bit of target.

    bits = target.bit_length()  # e.g. target=5 → bits=3, binary=101

    # We initialise reg_scratch = 1 (via reg_one + reg_one? No, that gives 2).
    # Cleaner restart: initialise reg_scratch = reg_one (value 1), then loop.
    # Reset scratch to 1 properly: scratch = reg_one (using +0? No zero allowed).
    # Trick: scratch = reg_one / reg_one = 1 (self-division trick).
    # But reg_scratch != reg_one, so: scratch = reg_one / reg_one.
    instructions.pop()  # remove the bad instruction we just added
    instructions.append(Instruction(op="/", src1=reg_one, src2=reg_one, dst=reg_scratch))
    # reg_scratch = 1 ✓

    # Process bits from MSB-1 down to LSB
    for bit_pos in range(bits - 2, -1, -1):
        # Double
        instructions.append(Instruction(op="+", src1=reg_scratch, src2=reg_scratch, dst=reg_scratch))
        # Add 1 if this bit is set
        if (target >> bit_pos) & 1:
            instructions.append(Instruction(op="+", src1=reg_scratch, src2=reg_one, dst=reg_scratch))

# ===========================================================================
# MODULE: src.solvers.tiny_bfs.py
# ===========================================================================
"""
src/solvers/tiny_bfs.py
Exact BFS solver for puzzles with small value_limit (< 100).

The 16 tiny scored puzzles (vl ≤ 16) have state spaces small enough for BFS
to find the guaranteed-optimal solution in milliseconds.

State: frozenset of integer values currently in the register pool.
       Order is irrelevant for reachability (we assign registers later).
Goal:  all targets are a subset of the state.

The BFS explores states by adding one instruction (one new value) at a time.
"""
from collections import deque
# [bundled] from src.utils import Puzzle, Instruction, apply_op, VALID_OPS


def tiny_bfs_solve(puzzle: Puzzle) -> list[Instruction]:
    """
    Find the minimum-instruction program for `puzzle` by BFS.
    Returns the optimal program. Falls back to the fallback solver on timeout
    or if value_limit is too large for practical BFS.
    """
    # Only run BFS when the state space is small
    if puzzle.value_limit > 500:
# [bundled]         from src.solvers import fallback
        return fallback.solve(puzzle)

    result = _bfs(puzzle)
    if result is not None:
        return result

    # BFS failed (shouldn't happen for tiny puzzles) — use fallback
# [bundled]     from src.solvers import fallback
    return fallback.solve(puzzle)


# ---------------------------------------------------------------------------
# BFS core
# ---------------------------------------------------------------------------

def _bfs(puzzle: Puzzle) -> list[Instruction] | None:
    """
    BFS over register-pool states.
    Returns list[Instruction] for the first (shortest) path found, or None.
    """
    seeds = tuple(puzzle.seeds)
    targets = frozenset(puzzle.targets)
    value_limit = puzzle.value_limit
    n_seeds = len(seeds)

    start_state = frozenset(seeds)
    if targets <= start_state:
        return []  # all targets are seeds — zero instructions

    # BFS: state = frozenset of live values
    # We track the path as a list of ValueOps to reconstruct instructions.
    # (op, a_val, b_val) → new_value

    # Queue: (state: frozenset, path: tuple of (op, a, b, result))
    queue = deque([(start_state, ())])
    visited = {start_state}

    while queue:
        state, path = queue.popleft()
        state_list = sorted(state)

        for a in state_list:
            for b in state_list:
                for op in VALID_OPS:
                    r = apply_op(op, a, b, value_limit)
                    if r is None or r in state:
                        continue

                    new_state = state | {r}
                    new_path = path + ((op, a, b, r),)

                    if targets <= new_state:
                        # Found optimal path
                        return _path_to_instructions(new_path, seeds)

                    if new_state not in visited:
                        visited.add(new_state)
                        queue.append((new_state, new_path))

    return None


def _path_to_instructions(
    path: tuple,
    seeds: tuple[int, ...],
) -> list[Instruction]:
    """
    Convert a sequence of (op, a_val, b_val, result_val) tuples to
    a valid list[Instruction] with concrete register assignments.

    Register layout:
      r0..r(K-1)  = seeds (fixed)
      r_K, r_K+1, ... = intermediates in order of computation
    """
    n_seeds = len(seeds)
    val_to_reg: dict[int, int] = {v: i for i, v in enumerate(seeds)}
    next_reg = n_seeds
    instructions: list[Instruction] = []

    for op, a_val, b_val, result_val in path:
        src1 = val_to_reg[a_val]
        src2 = val_to_reg[b_val]

        # Assign destination register
        if result_val in val_to_reg:
            dst = val_to_reg[result_val]
        elif next_reg < 48:
            dst = next_reg
            next_reg += 1
        else:
            # Register pressure (shouldn't happen in tiny puzzles)
            # Overwrite the oldest non-seed register
            dst = n_seeds  # simplistic
            # Remove old mapping
            for v, r in list(val_to_reg.items()):
                if r == dst:
                    del val_to_reg[v]
                    break

        val_to_reg[result_val] = dst
        instructions.append(Instruction(op=op, src1=src1, src2=src2, dst=dst))

    return instructions

# ===========================================================================
# MODULE: src.solvers.meet_in_middle.py
# ===========================================================================
"""
src/solvers/meet_in_middle.py
Pool-based meet-in-the-middle solver. The workhorse for the majority of puzzles.

Architecture:
  - Maintains a live pool of values ↔ register slots (max 48).
  - For each target, searches for the cheapest derivation path at depth 0–3
    using a hash-based backward lookup over the 1-op frontier set F.
  - Targets are processed in a chosen order that maximises pool reuse.
  - Falls back to binary construction for any target it cannot reach in ≤ 3 ops.

The pool tracks {value → register_index}. Seeds always occupy r0..r(K-1) and
are never evicted. As new values are computed, they occupy r_K onward.
When all 48 registers are live, we evict the value whose future utility is
the lowest (fewest remaining targets it participates in reaching).
"""
import random
from typing import Optional
# [bundled] from src.utils import (
# [bundled-cont]     Puzzle, Instruction, ValueOp, apply_op, build_one_op_map, VALID_OPS
# [bundled-cont] )


# ---------------------------------------------------------------------------
# LivePool: register ↔ value mapping
# ---------------------------------------------------------------------------

class LivePool:
    """
    Maps values ↔ register indices. Max size = 48.
    Seeds are pinned and never evicted.
    """
    MAX_REGS = 48

    def __init__(self, seeds: list[int]) -> None:
        self._val_to_reg: dict[int, int] = {}
        self._reg_to_val: dict[int, int] = {}
        self._seeds: frozenset[int] = frozenset(seeds)
        for i, v in enumerate(seeds):
            self._val_to_reg[v] = i
            self._reg_to_val[i] = v
        self._next_free: int = len(seeds)

    # -- queries --
    @property
    def values(self) -> set[int]:
        return set(self._val_to_reg)

    @property
    def size(self) -> int:
        return len(self._val_to_reg)

    def has(self, v: int) -> bool:
        return v in self._val_to_reg

    def reg_of(self, v: int) -> int:
        return self._val_to_reg[v]

    # -- mutations --
    def add_value(self, val: int, remaining_targets: set[int],
                  protect_regs: set[int] | None = None) -> int:
        """
        Add `val` to pool, evicting if full. Returns the dst register index.
        If val is already in pool, returns its current register (no-op).
        `protect_regs`: register indices that must NOT be evicted (e.g. operand regs).
        """
        if val in self._val_to_reg:
            return self._val_to_reg[val]

        if self._next_free < self.MAX_REGS:
            dst = self._next_free
            self._next_free += 1
        else:
            dst = self._evict(remaining_targets, protect_regs=protect_regs)

        self._val_to_reg[val] = dst
        self._reg_to_val[dst] = val
        return dst

    def _evict(self, remaining_targets: set[int],
               protect_regs: set[int] | None = None) -> int:
        """
        Choose and evict a non-seed value. Eviction policy:
        Score each non-seed value by how many remaining targets it can help
        reach in 1 op (higher = more useful = keep). Evict lowest-score.
        Tie-break: evict the value in the highest register index (LIFO).
        Returns the freed register index.
        protect_regs: register indices that must NOT be chosen for eviction.
        """
        protect_regs = protect_regs or set()
        best_reg = -1
        best_score = float("inf")

        for reg, val in self._reg_to_val.items():
            if val in self._seeds:
                continue  # never evict seeds
            if reg in protect_regs:
                continue  # protect operand registers
            # Utility: count remaining targets reachable from this value in 1 op
            score = 0
            for t in remaining_targets:
                if val == t:
                    score += 10  # it IS a target — very high utility
                    continue
                if t % val == 0 or val % t == 0:
                    score += 2
                diff = abs(t - val)
                if diff < val * 0.01:  # very close
                    score += 1

            if score < best_score or (score == best_score and reg > best_reg):
                best_score = score
                best_reg = reg

        if best_reg == -1:
            # All non-seed regs are protected — evict ANY non-seed (last resort)
            for reg, val in self._reg_to_val.items():
                if val not in self._seeds:
                    best_reg = reg
                    break
        if best_reg == -1:
            raise RuntimeError("Cannot evict: pool entirely occupied by seeds")

        evicted_val = self._reg_to_val[best_reg]
        del self._val_to_reg[evicted_val]
        del self._reg_to_val[best_reg]
        return best_reg

    def emit(self, op: str, a_val: int, b_val: int,
             result_val: int, remaining_targets: set[int]) -> Instruction:
        """
        Emit an instruction that computes `result_val = a_val op b_val`,
        updating the pool in place. Returns the Instruction.
        Reads src registers BEFORE add_value so eviction cannot clobber them.
        """
        src1 = self._val_to_reg[a_val]
        src2 = self._val_to_reg[b_val]
        # Pass protect_regs so eviction never frees src1 or src2
        dst = self.add_value(result_val, remaining_targets,
                             protect_regs={src1, src2})
        return Instruction(op=op, src1=src1, src2=src2, dst=dst)

    def emit_path(self, path: list, remaining_targets: set[int]) -> list:
        """
        Emit a sequence of ValueOps as Instructions, protecting ALL future
        operand values from eviction at each step.
        Returns list[Instruction].
        """
        instructions = []
        for step_i, vop in enumerate(path):
            # Collect all values needed by future steps in this path
            future_vals: set[int] = set()
            for future_vop in path[step_i + 1:]:
                future_vals.add(future_vop.a)
                future_vals.add(future_vop.b)
            # Map future values to current registers (only those in pool)
            future_regs = {
                self._val_to_reg[v]
                for v in future_vals
                if v in self._val_to_reg
            }
            src1 = self._val_to_reg[vop.a]
            src2 = self._val_to_reg[vop.b]
            protect = {src1, src2} | future_regs
            dst = self.add_value(vop.result, remaining_targets,
                                 protect_regs=protect)
            instructions.append(Instruction(op=vop.op, src1=src1, src2=src2, dst=dst))
        return instructions


# ---------------------------------------------------------------------------
# Search: find_target
# ---------------------------------------------------------------------------

def find_target(
    target: int,
    pool: LivePool,
    value_limit: int,
    max_depth: int = 3,
) -> Optional[list[ValueOp]]:
    """
    Find a path of ValueOps to produce `target` using values in `pool`.
    Returns a list (possibly empty for depth-0 hits) or None if not found.

    The returned list has depth = number of new instructions required:
      depth 0 → target already in pool
      depth 1 → one instruction
      depth 2 → two instructions (build one intermediate, combine)
      depth 3 → three instructions (build two intermediates, combine)
    """
    pool_vals = pool.values

    # --- Depth 0: already in pool ---
    if target in pool_vals:
        return []

    pool_list = sorted(pool_vals)

    # --- Depth 1: target = pool_a op pool_b ---
    for a in pool_list:
        for b in pool_list:
            for op in VALID_OPS:
                r = apply_op(op, a, b, value_limit)
                if r == target:
                    return [ValueOp(op=op, a=a, b=b, result=target)]

    if max_depth < 2:
        return None

    # Build 1-op frontier F = all values reachable in one op from pool
    F: dict[int, ValueOp] = build_one_op_map(pool_vals, value_limit)
    all_reachable = pool_vals | set(F.keys())

    # --- Depth 2 & 3: target = f op x (or x op f) ---
    # For each f in F, compute backward candidates x such that f op x = target
    # or x op f = target. Check if x is in pool (depth 2) or F (depth 3).
    for f_val, f_vop in F.items():
        for path in _backward_candidates(target, f_val, all_reachable, value_limit, F, pool_vals):
            build_f = f_vop
            full_path = [build_f] + path
            if 0 < len(full_path) <= max_depth:
                # Validate: replay the path and ensure every operand is available
                if _path_operands_ok(full_path, pool_vals):
                    return full_path

    return None


def _path_operands_ok(path: list, pool_vals: set[int]) -> bool:
    """
    Check that every step in `path` can be executed in order:
    - Both operands (a, b) must be in pool_vals or produced by an earlier step.
    Returns True if safe, False if any operand is missing.
    """
    available = set(pool_vals)
    for vop in path:
        if vop.a not in available or vop.b not in available:
            return False
        available.add(vop.result)
    return True


def _backward_candidates(
    target: int,
    f_val: int,
    all_reachable: set[int],
    value_limit: int,
    F: dict[int, ValueOp],
    pool_vals: set[int],
) -> list[list[ValueOp]]:
    """
    For a given f_val already computed (from F), find all x in all_reachable
    such that f_val op x = target or x op f_val = target.
    Returns paths (not including the step to build f_val).
    """
    results = []
    f_vop = F[f_val]  # the ValueOp that produced f_val (used in final path)

    def try_x(x: int, combine_op: str, operand_order: str) -> None:
        """Check if x is reachable and add a path if so."""
        if x <= 0 or x > value_limit:
            return
        if x not in all_reachable:
            return
        # Verify combine step produces target
        if operand_order == "fx":
            check = apply_op(combine_op, f_val, x, value_limit)
        else:
            check = apply_op(combine_op, x, f_val, value_limit)
        if check != target:
            return

        # Build path: [build_x (if x in F not pool), combine_to_target]
        if x in pool_vals:
            # depth 2 total (f + combine = 2 ops; we're generating the 2nd part)
            if operand_order == "fx":
                combine = ValueOp(op=combine_op, a=f_val, b=x, result=target)
            else:
                combine = ValueOp(op=combine_op, a=x, b=f_val, result=target)
            results.append([combine])
        elif x in F:
            # depth 3 total (f + build_x + combine = 3 ops)
            build_x = F[x]
            if operand_order == "fx":
                combine = ValueOp(op=combine_op, a=f_val, b=x, result=target)
            else:
                combine = ValueOp(op=combine_op, a=x, b=f_val, result=target)
            results.append([build_x, combine])

    # target = f + x → x = target - f
    try_x(target - f_val, "+", "fx")
    # target = x + f (same result, f+x commutative; skip to avoid duplicates)
    # target = f - x → x = f - target  (need f > target)
    if f_val > target:
        try_x(f_val - target, "-", "fx")
    # target = x - f → x = target + f
    try_x(target + f_val, "-", "xf")
    # target = f * x → x = target // f (if exact)
    if f_val != 0 and target % f_val == 0:
        try_x(target // f_val, "*", "fx")
    # target = f / x → x = f // target (if f divisible by target)
    if target != 0 and f_val % target == 0:
        try_x(f_val // target, "/", "fx")
    # target = x / f → x = target * f
    x_div = target * f_val
    if x_div <= value_limit:
        try_x(x_div, "/", "xf")

    return results


# ---------------------------------------------------------------------------
# Target ordering
# ---------------------------------------------------------------------------

def order_targets(
    targets: list[int],
    pool: LivePool,
    value_limit: int,
) -> list[int]:
    """
    Order targets to maximise pool reuse.
    Priority:
      1. Targets already in pool (cost 0).
      2. Targets reachable in 1 op from current pool.
      3. Smooth targets (small prime factors → cheaper to build).
      4. Remaining targets in ascending order of magnitude (heuristic).
    """
    import math
    pool_vals = pool.values
    F = build_one_op_map(pool_vals, value_limit)

    free    = [t for t in targets if t in pool_vals]
    depth1  = [t for t in targets if t not in pool_vals and t in F]
    rest    = [t for t in targets if t not in pool_vals and t not in F]

    # Sort rest: smooth-first, then ascending
    def sort_key(t):
        # Approximate smoothness: count small factors
        n, smooth_score = t, 0
        for p in [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47]:
            while n % p == 0:
                n //= p
                smooth_score += 1
        return (-smooth_score, t)

    rest.sort(key=sort_key)
    return free + depth1 + rest


# ---------------------------------------------------------------------------
# Main solver
# ---------------------------------------------------------------------------

def meet_in_middle_solve(puzzle: Puzzle, *, random_seed: Optional[int] = None) -> list[Instruction]:
    """
    Solve a puzzle using pool-based meet-in-the-middle search.
    Returns a valid list[Instruction] (falls back to binary construction
    for any target that can't be reached in ≤ 3 ops).

    `random_seed`: if set, adds a small random perturbation to target ordering
    for use in multi-start ensembles.
    """
# [bundled]     from src.solvers.fallback import _build_target as _fallback_build

    pool = LivePool(puzzle.seeds)
    instructions: list[Instruction] = []
    remaining = set(puzzle.targets)

    # Satisfy seed-equal targets for free
    for s in puzzle.seeds:
        remaining.discard(s)

    if not remaining:
        return instructions

    # Produce constant 1 (needed by fallback sub-paths) if any seed ≥ 2
    # We'll only produce it if we actually need it.
    have_one = 1 in pool.values
    reg_one: Optional[int] = pool.reg_of(1) if have_one else None

    # Need a scratch register for fallback targets
    # Reserve r(n_seeds) for constant-1 and r(n_seeds+1) for scratch if needed.
    reg_fallback_scratch = puzzle.registers - 1  # r47: last register, used as scratch

    ordered = order_targets(list(remaining), pool, puzzle.value_limit)

    if random_seed is not None:
        rng = random.Random(random_seed)
        # Lightly shuffle only the non-free, non-depth1 portion (rest)
        # Count free (depth-0) and depth-1 targets at the front
        pool_vals_now = pool.values
        F_now = build_one_op_map(pool_vals_now, puzzle.value_limit)
        fixed_count = sum(
            1 for t in ordered
            if t in pool_vals_now or t in F_now
        )
        if fixed_count < len(ordered):
            tail = ordered[fixed_count:]
            rng.shuffle(tail)
            ordered = ordered[:fixed_count] + tail

    for target in ordered:
        if not pool.has(target):
            # Try M-i-M search (depth ≤ 3)
            path = find_target(target, pool, puzzle.value_limit, max_depth=3)

            if path is not None:
                # Emit the value ops as register instructions
                # Use emit_path to protect future operands from eviction
                instructions.extend(pool.emit_path(path, remaining))
            else:
                # Fallback: binary construction for this target
                # We need reg_one (constant 1) in the pool
                if not pool.has(1):
                    # Produce 1 using seeds[0] / seeds[0]
                    seed0 = puzzle.seeds[0]
                    inst = pool.emit("/", seed0, seed0, 1, remaining)
                    instructions.append(inst)

                # Build target into a fresh/scratch register
                _emit_binary_target(target, pool, instructions, remaining,
                                    puzzle.value_limit)

        remaining.discard(target)

    return instructions


def _emit_binary_target(
    target: int,
    pool: LivePool,
    instructions: list[Instruction],
    remaining: set[int],
    value_limit: int,
) -> None:
    """
    Emit binary construction instructions for `target` using the pool.
    Requires that 1 is already in the pool.
    Tracks all intermediate values in the pool.
    """
    if target == 1:
        return  # 1 is already in pool

    one = 1  # pool must have this
    bits = target.bit_length()

    # Start accumulator at 1
    # We'll track the accumulator value and emit into pool
    accum = 1

    for bit_pos in range(bits - 2, -1, -1):
        # Double
        new_accum = accum * 2
        if new_accum > value_limit:
            # Shouldn't happen (target ≤ value_limit and accum ≤ target/2)
            # but guard anyway — fall through without completing (caller handles)
            return
        inst = pool.emit("+", accum, accum, new_accum, remaining)
        instructions.append(inst)
        accum = new_accum

        # Add 1 if bit is set
        if (target >> bit_pos) & 1:
            new_accum = accum + 1
            if new_accum > value_limit:
                return
            inst = pool.emit("+", accum, one, new_accum, remaining)
            instructions.append(inst)
            accum = new_accum

    # If we didn't reach target (e.g. value_limit overflow mid-build), that
    # is a caller issue — the verifier will catch it. For safety, assert here
    # during development but don't raise in production.
    # assert accum == target, f"_emit_binary_target failed: got {accum}, want {target}"

# ===========================================================================
# MODULE: src.solvers.delta_cluster.py
# ===========================================================================
"""
src/solvers/delta_cluster.py
Cluster-aware solver for puzzles where many targets are close together.

Strategy:
  1. Sort targets and find clusters: runs where adjacent targets are within
     rel_tol of each other.
  2. For each cluster, pick the median value as an anchor and solve it using
     the M-i-M core.
  3. Each other member is derived as: anchor ± delta, where delta is computed
     cheaply (usually 1 instruction if delta is reachable from pool).
  4. Non-clustered targets are solved by the M-i-M core directly.

Expected saving: 1–2 instructions per additional cluster member (vs. M-i-M alone).
"""
# [bundled] from src.utils import Puzzle, Instruction, apply_op
# [bundled] from src.solvers.meet_in_middle import LivePool, find_target


REL_TOL = 0.005  # 0.5% — targets within this ratio are considered clustered


def _find_clusters(targets: list[int]) -> list[list[int]]:
    """
    Group sorted targets into clusters where adjacent members are within REL_TOL.
    Returns a list of cluster groups (each group is a sorted list of targets).
    Single-element groups are still returned (non-clustered).
    """
    if not targets:
        return []
    t = sorted(targets)
    clusters: list[list[int]] = []
    current = [t[0]]
    for i in range(1, len(t)):
        ref = current[0]
        if ref > 0 and abs(t[i] - ref) / ref <= REL_TOL:
            current.append(t[i])
        else:
            clusters.append(current)
            current = [t[i]]
    clusters.append(current)
    return clusters


def _solve_via_mim(
    target: int,
    pool: LivePool,
    instructions: list[Instruction],
    remaining: set[int],
    puzzle: "Puzzle",
) -> None:
    """Solve one target via M-i-M or fallback binary construction."""
    if pool.has(target):
        return
    path = find_target(target, pool, puzzle.value_limit, max_depth=3)
    if path is not None:
        instructions.extend(pool.emit_path(path, remaining))
    else:
        _dc_fallback_target(target, pool, instructions, remaining, puzzle)


def delta_cluster_solve(puzzle: Puzzle) -> list[Instruction]:
    """
    Solve using cluster detection + M-i-M for anchors + delta derivation.
    """
    pool = LivePool(puzzle.seeds)
    instructions: list[Instruction] = []
    remaining = set(puzzle.targets)

    # Mark seeds that equal targets as free
    for s in puzzle.seeds:
        remaining.discard(s)

    clusters = _find_clusters(list(puzzle.targets))

    for cluster in clusters:
        if len(cluster) == 1:
            t = cluster[0]
            _solve_via_mim(t, pool, instructions, remaining, puzzle)
            remaining.discard(t)
        else:
            # Multi-target cluster: build anchor then derive others by delta
            anchor = cluster[len(cluster) // 2]  # median

            # --- Solve anchor ---
            _solve_via_mim(anchor, pool, instructions, remaining, puzzle)
            remaining.discard(anchor)

            # --- Derive cluster members from anchor ---
            for t in cluster:
                if t == anchor:
                    continue
                if pool.has(t):
                    remaining.discard(t)
                    continue

                delta = abs(t - anchor)
                if delta == 0:
                    remaining.discard(t)
                    continue

                derived = False

                # Strategy 1: delta already in pool → 1 instruction
                if pool.has(delta):
                    op = "+" if t > anchor else "-"
                    r = apply_op(op, anchor, delta, puzzle.value_limit)
                    if r == t:
                        instructions.append(pool.emit(op, anchor, delta, t, remaining))
                        derived = True

                # Strategy 2: build delta via M-i-M (depth ≤ 2), then combine
                if not derived and delta <= puzzle.value_limit:
                    path_delta = find_target(delta, pool, puzzle.value_limit, max_depth=2)
                    if path_delta is not None:
                        # Extend path_delta with the combine step
                        op = "+" if t > anchor else "-"
                        r = apply_op(op, anchor, delta, puzzle.value_limit)
                        if r == t:
# [bundled]                             from src.utils import ValueOp
                            combine_vop = ValueOp(op=op, a=anchor, b=delta, result=t)
                            full_path = path_delta + [combine_vop]
                            instructions.extend(pool.emit_path(full_path, remaining))
                            derived = True

                # Strategy 3: solve t directly with M-i-M or fallback
                if not derived:
                    _solve_via_mim(t, pool, instructions, remaining, puzzle)

                remaining.discard(t)

    return instructions


def _dc_fallback_target(
    target: int,
    pool: LivePool,
    instructions: list[Instruction],
    remaining: set[int],
    puzzle: Puzzle,
) -> None:
    """Emit binary construction for a single target using the pool."""
# [bundled]     from src.solvers.meet_in_middle import _emit_binary_target
    if not pool.has(1):
        s0 = puzzle.seeds[0]
        inst = pool.emit("/", s0, s0, 1, remaining)
        instructions.append(inst)
    _emit_binary_target(target, pool, instructions, remaining, puzzle.value_limit)

# ===========================================================================
# MODULE: src.solvers.factor_smooth.py
# ===========================================================================
"""
src/solvers/factor_smooth.py
Factor-decomposition solver for puzzles where many targets have small prime factors.

Strategy:
  1. Factor each target using trial division up to 2000 (covering smooth targets).
  2. Build a "factor basis": prime factors ≤ 2000 that appear across the puzzle's
     smooth targets. Build each basis prime from the seeds using M-i-M.
  3. For each smooth target, express it as a product/combination of basis elements:
       t = a * b (if both live in pool) → 1 instruction
       t = a * b ± c                  → 2–3 instructions
  4. Non-smooth targets fall back to M-i-M core.
"""
# [bundled] from src.utils import Puzzle, Instruction, apply_op
# [bundled] from src.solvers.meet_in_middle import LivePool, find_target


def _trial_factor(n: int, limit: int = 2000) -> tuple[dict[int, int], int]:
    """
    Trial division up to `limit`. Returns (prime_exponents_dict, remainder).
    If remainder == 1, n is smooth (all factors ≤ limit).
    """
    factors: dict[int, int] = {}
    for p in range(2, min(limit + 1, int(n**0.5) + 1)):
        while n % p == 0:
            factors[p] = factors.get(p, 0) + 1
            n //= p
    return factors, n  # remainder n (if > 1, it's a prime > limit)


def _collect_basis_primes(targets: list[int]) -> set[int]:
    """Collect all small prime factors (≤ 2000) across all targets."""
    primes: set[int] = set()
    for t in targets:
        factors, _ = _trial_factor(t)
        primes.update(factors.keys())
    return primes


def factor_smooth_solve(puzzle: Puzzle) -> list[Instruction]:
    """
    Solve using factor decomposition for smooth targets; M-i-M for the rest.
    """
    pool = LivePool(puzzle.seeds)
    instructions: list[Instruction] = []
    remaining = set(puzzle.targets)

    # Mark seeds that equal targets as free
    for s in puzzle.seeds:
        remaining.discard(s)

    # --- Phase 1: build the factor basis from seeds ---
    basis_primes = _collect_basis_primes(list(remaining))
    # Remove primes already in pool (seeds)
    basis_to_build = [p for p in sorted(basis_primes) if not pool.has(p)]

    for prime in basis_to_build:
        if prime > puzzle.value_limit:
            continue  # can't build this prime; skip
        path = find_target(prime, pool, puzzle.value_limit, max_depth=3)
        if path is not None:
            instructions.extend(pool.emit_path(path, remaining))

    # --- Phase 2: process each target ---
    for target in sorted(remaining):  # ascending order
        if pool.has(target):
            remaining.discard(target)
            continue

        factors, remainder = _trial_factor(target)

        if remainder == 1:
            # Fully smooth: express as product of prime powers in pool
            derived = _build_from_factors(target, factors, pool, instructions, remaining, puzzle.value_limit)
        else:
            derived = False

        if not derived:
            # Fall back to M-i-M (which may itself fall back to binary)
            path = find_target(target, pool, puzzle.value_limit, max_depth=3)
            if path is not None:
                instructions.extend(pool.emit_path(path, remaining))
            else:
                _fs_fallback_target(target, pool, instructions, remaining, puzzle)

        remaining.discard(target)

    return instructions


def _build_from_factors(
    target: int,
    factors: dict[int, int],
    pool: LivePool,
    instructions: list[Instruction],
    remaining: set[int],
    value_limit: int,
) -> bool:
    """
    Try to build `target` by multiplying together its prime powers.
    Uses values currently in pool. Returns True if successful.
    """
    # Build prime powers that aren't yet in pool
    for p, exp in factors.items():
        power = p ** exp
        if power > value_limit:
            return False
        if not pool.has(power):
            # Try to build p^exp from p (if p is in pool)
            if pool.has(p):
                current = p
                for _ in range(exp - 1):
                    new_val = current * p
                    if new_val > value_limit:
                        return False
                    r = apply_op("*", current, p, value_limit)
                    if r is None:
                        return False
                    instructions.append(pool.emit("*", current, p, new_val, remaining))
                    current = new_val
            else:
                return False  # prime not in pool, can't build

    # Now multiply all prime powers together
    # Collect all prime powers
    prime_powers = []
    for p, exp in sorted(factors.items()):
        prime_powers.append(p ** exp)

    if not all(pool.has(pw) for pw in prime_powers):
        return False

    if len(prime_powers) == 1:
        # Only one prime power — it must equal target
        return prime_powers[0] == target

    # Multiply them pairwise
    accum = prime_powers[0]
    for pw in prime_powers[1:]:
        new_val = accum * pw
        if new_val > value_limit:
            return False
        r = apply_op("*", accum, pw, value_limit)
        if r is None:
            return False
        instructions.append(pool.emit("*", accum, pw, new_val, remaining))
        accum = new_val

    return accum == target


def _fs_fallback_target(
    target: int,
    pool: LivePool,
    instructions: list[Instruction],
    remaining: set[int],
    puzzle: Puzzle,
) -> None:
# [bundled]     from src.solvers.meet_in_middle import _emit_binary_target
    if not pool.has(1):
        s0 = puzzle.seeds[0]
        instructions.append(pool.emit("/", s0, s0, 1, remaining))
    _emit_binary_target(target, pool, instructions, remaining, puzzle.value_limit)

# ===========================================================================
# MODULE: src.solvers.binary_shared.py
# ===========================================================================
"""
src/solvers/binary_shared.py
Binary construction with shared-prefix optimization + delta pass.

THE CORE INSIGHT:
  Seeds are ~10^3, targets are ~10^14-10^18 — a gap of ~36 doublings.
  Depth-3 M-i-M from seeds NEVER reaches target magnitude.
  
  The correct approach: binary construction is the backbone. Savings come from:
  1. SHARED PREFIXES: If two targets share MSBs, build the prefix once.
  2. POOL REUSE: When building target T, if pool already has V = T>>k,
     start from V instead of 1 — saves k doublings.
  3. FREE INTERMEDIATES: Values produced mid-chain that ARE targets cost 0.
  4. DELTA PASS: After binary build, derive remaining targets cheaply
     from pool values using ±1, *2, or small multiples.
  5. MULTIPLICATION: T = V * 2^k is just k doublings of V.
"""
# [bundled] from src.utils import Puzzle, Instruction, apply_op
# [bundled] from src.solvers.meet_in_middle import LivePool


# ---------------------------------------------------------------------------
# Core: find best pool prefix for a target
# ---------------------------------------------------------------------------

def _find_best_prefix(target: int, pool: LivePool) -> tuple[int, int]:
    """
    Find the pool value V that covers the most MSBs of target.
    Returns (V, k) where target >> k == V and k is minimized.
    Falls back to (1, bit_length-1) if nothing better found.
    """
    best_v = 1
    best_k = target.bit_length() - 1  # starting from 1
    t_bits = target.bit_length()

    for v in pool.values:
        if v <= 0 or v > target:
            continue
        v_bits = v.bit_length()
        k = t_bits - v_bits
        if k < 0:
            continue
        # Check exact prefix match: target >> k == v
        if (target >> k) == v and k < best_k:
            best_k = k
            best_v = v

    return best_v, best_k


# ---------------------------------------------------------------------------
# Core: build one target from a given starting value
# ---------------------------------------------------------------------------

def _build_from_value(
    target: int,
    start_v: int,
    start_k: int,
    pool: LivePool,
    instructions: list[Instruction],
    remaining: set[int],
    value_limit: int,
) -> None:
    """
    Build `target` by extending from start_v using binary construction.
    Processes bits (start_k-1) down to 0.
    """
    accum = start_v
    for bit_pos in range(start_k - 1, -1, -1):
        # Double
        new_accum = accum * 2
        if new_accum > value_limit:
            return  # safety guard
        inst = pool.emit("+", accum, accum, new_accum, remaining)
        instructions.append(inst)
        accum = new_accum

        # Add 1 if this bit is set
        if (target >> bit_pos) & 1:
            new_accum = accum + 1
            if new_accum > value_limit:
                return
            inst = pool.emit("+", accum, 1, new_accum, remaining)
            instructions.append(inst)
            accum = new_accum


def _build_one_target(
    target: int,
    pool: LivePool,
    instructions: list[Instruction],
    remaining: set[int],
    value_limit: int,
) -> None:
    """Build target using pool's best available prefix."""
    if pool.has(target):
        return
    start_v, start_k = _find_best_prefix(target, pool)
    if start_k == 0:
        return  # target is already in pool as a prefix (shouldn't happen)
    _build_from_value(target, start_v, start_k, pool, instructions, remaining, value_limit)


# ---------------------------------------------------------------------------
# Delta pass: derive remaining targets cheaply from pool
# ---------------------------------------------------------------------------

def _delta_pass(
    remaining: set[int],
    pool: LivePool,
    instructions: list[Instruction],
    value_limit: int,
) -> set[int]:
    """
    Try to derive remaining targets from pool values using 1 instruction:
    - T = V + 1 (if 1 in pool)
    - T = V - 1 (if 1 in pool and V > 1)  
    - T = V * 2 (doubling)
    - T = V + W (for W in pool)
    - T = V - W (for W in pool, V > W)
    Returns the set of targets still not satisfied.
    """
    still_remaining = set(remaining)
    pool_list = sorted(pool.values)

    changed = True
    while changed and still_remaining:
        changed = False
        for t in sorted(still_remaining):
            if pool.has(t):
                still_remaining.discard(t)
                changed = True
                continue
            # Try 1-op derivation
            derived = False
            for v in pool_list:
                # T = V + W
                w = t - v
                if w > 0 and pool.has(w):
                    r = apply_op("+", v, w, value_limit)
                    if r == t:
                        instructions.append(pool.emit("+", v, w, t, still_remaining))
                        derived = True
                        break
                # T = V * 2 (doubling)
                if v * 2 == t and t <= value_limit:
                    r = apply_op("+", v, v, value_limit)
                    if r == t:
                        instructions.append(pool.emit("+", v, v, t, still_remaining))
                        derived = True
                        break
                # T = V - W
                if v > t:
                    w = v - t
                    if w > 0 and pool.has(w):
                        r = apply_op("-", v, w, value_limit)
                        if r == t:
                            instructions.append(pool.emit("-", v, w, t, still_remaining))
                            derived = True
                            break
                # T = V / W  
                if v > 0 and v % t == 0:
                    w = v // t
                    if w > 0 and pool.has(w):
                        r = apply_op("/", v, w, value_limit)
                        if r == t:
                            instructions.append(pool.emit("/", v, w, t, still_remaining))
                            derived = True
                            break
                # T = V * W  
                if t > v and v > 0 and t % v == 0:
                    w = t // v
                    if w > 0 and pool.has(w):
                        r = apply_op("*", v, w, value_limit)
                        if r == t:
                            instructions.append(pool.emit("*", v, w, t, still_remaining))
                            derived = True
                            break

            if derived:
                still_remaining.discard(t)
                changed = True
                # Refresh pool_list since pool changed
                pool_list = sorted(pool.values)

    return still_remaining


# ---------------------------------------------------------------------------
# Main solver
# ---------------------------------------------------------------------------

def binary_shared_solve(puzzle: Puzzle) -> list[Instruction]:
    """
    Solve using pool-aware binary construction with prefix sharing + delta pass.
    """
    pool = LivePool(puzzle.seeds)
    instructions: list[Instruction] = []
    remaining = set(puzzle.targets)

    # Mark seeds that equal targets as free
    for s in puzzle.seeds:
        remaining.discard(s)

    if not remaining:
        return instructions

    # --- Produce constant 1 (needed for binary construction) ---
    if not pool.has(1):
        s0 = puzzle.seeds[0]
        inst = pool.emit("/", s0, s0, 1, remaining)
        instructions.append(inst)

    # --- Phase 1: Delta pass (find anything derivable in 1 op from seeds) ---
    remaining = _delta_pass(remaining, pool, instructions, puzzle.value_limit)

    # --- Phase 2: Binary construction with prefix sharing ---
    # Order: shorter bit-length first → builds up pool of prefixes
    ordered = sorted(remaining, key=lambda t: (t.bit_length(), t))

    for target in ordered:
        if pool.has(target):
            remaining.discard(target)
            continue
        _build_one_target(target, pool, instructions, remaining, puzzle.value_limit)
        remaining.discard(target)

        # After each build, run a quick delta pass on remaining
        # (the newly built target may enable 1-op derivation of others)
        if remaining:
            # Quick single-pass delta check: check only new derivations from target
            new_remaining = set()
            for t in sorted(remaining):
                if pool.has(t):
                    continue
                derived = False
                # T = target * 2?
                if target * 2 == t and t <= puzzle.value_limit:
                    r = apply_op("+", target, target, puzzle.value_limit)
                    if r == t:
                        instructions.append(pool.emit("+", target, target, t, remaining))
                        derived = True
                # T = target + 1?
                if not derived and pool.has(1):
                    r = apply_op("+", target, 1, puzzle.value_limit)
                    if r == t:
                        instructions.append(pool.emit("+", target, 1, t, remaining))
                        derived = True
                # T = target - 1?
                if not derived and pool.has(1) and target > 1:
                    r = apply_op("-", target, 1, puzzle.value_limit)
                    if r == t:
                        instructions.append(pool.emit("-", target, 1, t, remaining))
                        derived = True
                if not derived:
                    new_remaining.add(t)
            remaining = new_remaining

    return instructions

# ===========================================================================
# SOLVER NAMESPACE SHIMS
# (allow orchestrator to call fallback.solve(...) etc. in the flat namespace)
# ===========================================================================
class _NS:
    """Tiny namespace shim — exposes a single callable as .solve"""
    def __init__(self, fn): self.solve = fn

fallback       = _NS(fallback_solve)
tiny_bfs       = _NS(tiny_bfs_solve)
meet_in_middle = _NS(meet_in_middle_solve)
delta_cluster  = _NS(delta_cluster_solve)
factor_smooth  = _NS(factor_smooth_solve)
binary_shared  = _NS(binary_shared_solve)

# ===========================================================================
# MODULE: src.postpass.py
# ===========================================================================
"""
src/postpass.py
Post-processing passes applied after any solver produces a valid program.

Passes (applied in order):
  1. remove_dead_ops  — eliminates instructions whose output is never read
                        and doesn't hit a target. Free improvement, O(n).
  2. peephole        — 2–3 instruction window local rewrites. O(n).
  3. simulated_annealing — probabilistic local search over the instruction
                        sequence. The verifier is the oracle. Time-bounded.

Each pass takes a Puzzle + list[Instruction] → list[Instruction].
All passes produce programs that pass verify_program().
"""
import random
import time
# [bundled] from src.utils import Puzzle, Instruction, apply_op, VALID_OPS
# [bundled] from src.verifier import verify_program


# ---------------------------------------------------------------------------
# Pass 1: Dead Instruction Elimination
# ---------------------------------------------------------------------------

def remove_dead_ops(puzzle: Puzzle, program: list[Instruction]) -> list[Instruction]:
    """
    Remove instructions whose destination register is overwritten before it is
    ever read AND whose result value never satisfies a target.

    Algorithm (single backward scan):
      - Track which registers are "needed" at each point.
      - An instruction is live if its dst register is in the needed set at
        that point OR if its result value is in puzzle.targets.
      - After scanning backwards, re-filter to keep only live instructions.
    """
    if not program:
        return program

    n = len(program)
    targets = set(puzzle.targets)
    seeds = set(puzzle.seeds)

    # Forward simulation: track register values so we know what each
    # instruction produces (needed to check "result in targets").
    regs: list[int | None] = [None] * puzzle.registers
    for i, v in enumerate(puzzle.seeds):
        regs[i] = v

    result_values: list[int | None] = []
    for inst in program:
        a = regs[inst.src1]
        b = regs[inst.src2]
        r = apply_op(inst.op, a, b, puzzle.value_limit) if (a is not None and b is not None) else None
        result_values.append(r)
        if r is not None:
            regs[inst.dst] = r

    # Backward scan: which registers are needed AFTER step i?
    # A register is needed if it will be used as src1/src2 in a future instruction.
    needed: set[int] = set()

    # At the end, no register is needed (targets are recorded during simulation)
    live = [False] * n

    for i in range(n - 1, -1, -1):
        inst = program[i]
        result_val = result_values[i]

        # This instruction is live if:
        # (a) its dst is currently needed by a later instruction, OR
        # (b) its result is a target value (must have been produced)
        is_target_hit = result_val is not None and result_val in targets
        if inst.dst in needed or is_target_hit:
            live[i] = True
            # This instruction reads src1 and src2, so they become needed
            needed.discard(inst.dst)  # dst is now "satisfied" by this inst
            needed.add(inst.src1)
            needed.add(inst.src2)
        # If not live: dst is overwritten (or never read), skip

    return [program[i] for i in range(n) if live[i]]


# ---------------------------------------------------------------------------
# Pass 2: Peephole Rewrites
# ---------------------------------------------------------------------------

def peephole(puzzle: Puzzle, program: list[Instruction]) -> list[Instruction]:
    """
    Apply local rewrites. Currently implemented:
      1. Dead-op elimination (via remove_dead_ops).
      2. Consecutive-doubling → multiply: k doublings of register R
         ('+', R, R, R) repeated k times can be replaced by ('*', R, k_reg, R)
         if k is a power of 2 and k is in the pool at that point. Actually,
         k doublings = multiply by 2^k. If 2^k is reachable via multiplication
         by a constant already in pool, we might save ops. For now we do the
         dead-op pass which handles most of this.
    """
    # First do a full dead-op removal
    result = remove_dead_ops(puzzle, program)

    # Peephole: collapse runs of self-doublings (R = R + R) into a single multiply
    # Pattern: n consecutive (+, r, r, r) → try to find if 2^n is in pool at that point
    # This is complex to implement correctly with register simulation; skip for now.
    # The SA pass handles this probabilistically.
    return result


# ---------------------------------------------------------------------------
# Pass 3: Simulated Annealing
# ---------------------------------------------------------------------------

def simulated_annealing(
    puzzle: Puzzle,
    program: list[Instruction],
    *,
    time_budget: float = 2.0,
    initial_temp: float = 2.0,
    final_temp: float = 0.01,
    random_seed: int | None = None,
) -> list[Instruction]:
    """
    Probabilistic local search over the instruction sequence.

    Move types (weighted):
      1. TryRemove (60%): remove one random instruction; accept if still valid.
      2. TryRemoveBlock (20%): remove 2-3 consecutive instructions; accept if valid.
      3. TrySwap (10%): swap two instructions.
      4. TryReplaceOp (10%): change operator of one instruction.

    Returns the best program found within the time budget.
    """
    rng = random.Random(random_seed)
    best = list(program)
    best_count = len(best)

    if best_count == 0:
        return best

    current = list(best)
    current_count = best_count

    start = time.monotonic()

    while True:
        elapsed = time.monotonic() - start
        if elapsed >= time_budget:
            break

        # Geometric cooling
        frac = elapsed / time_budget
        temp = initial_temp * ((final_temp / initial_temp) ** frac)

        # --- Pick a random move ---
        r = rng.random()
        if r < 0.60 and current_count > 1:
            # TryRemove: remove one instruction
            idx = rng.randrange(current_count)
            candidate = current[:idx] + current[idx + 1:]
        elif r < 0.80 and current_count > 3:
            # TryRemoveBlock: remove 2-4 consecutive instructions
            block = rng.randint(2, min(4, current_count - 1))
            idx = rng.randrange(current_count - block)
            candidate = current[:idx] + current[idx + block:]
        elif r < 0.90 and current_count > 2:
            # TrySwap
            i, j = sorted(rng.sample(range(current_count), 2))
            candidate = list(current)
            candidate[i], candidate[j] = candidate[j], candidate[i]
        elif current_count > 0:
            # TryReplaceOp
            idx = rng.randrange(current_count)
            new_op = rng.choice(["+", "-", "*", "/"])
            inst = current[idx]
            candidate = list(current)
            candidate[idx] = Instruction(op=new_op, src1=inst.src1,
                                         src2=inst.src2, dst=inst.dst)
        else:
            continue

        # Validate candidate
        result = verify_program(puzzle, candidate)
        if result.valid:
            delta = len(candidate) - current_count
            if delta < 0:
                # Always accept improvements
                current = candidate
                current_count = len(candidate)
                if current_count < best_count:
                    best = candidate[:]
                    best_count = current_count
            elif delta == 0:
                # Accept neutral moves unconditionally (exploration)
                current = candidate
            # delta > 0: reject

    return best


# ---------------------------------------------------------------------------
# Combined post-pass pipeline
# ---------------------------------------------------------------------------

def postprocess(
    puzzle: Puzzle,
    program: list[Instruction],
    *,
    run_annealing: bool = True,
    annealing_budget: float = 2.0,
    random_seed: int | None = None,
) -> list[Instruction]:
    """
    Run the full post-processing pipeline on a valid program.
    Returns the (hopefully shorter) improved program.
    """
    # Pass 1: dead-op elimination
    program = remove_dead_ops(puzzle, program)

    # Pass 2: peephole
    program = peephole(puzzle, program)

    # Pass 3: simulated annealing (optional, time-bounded)
    if run_annealing and len(program) > 0:
        program = simulated_annealing(
            puzzle, program,
            time_budget=annealing_budget,
            random_seed=random_seed,
        )
        # Final dead-op pass after annealing
        program = remove_dead_ops(puzzle, program)

    return program

# ===========================================================================
# MODULE: src.orchestrator.py
# ===========================================================================
"""
src/orchestrator.py
Main driver: runs the solver portfolio per puzzle, picks the best program,
applies post-processing, and writes the final submission CSV.

Usage (CLI):
    python -m src.orchestrator \
        --puzzles  data/puzzles.csv \
        --out      submissions/sub_v1.csv \
        --cores    8 \
        --sa-time  2.0

Usage (programmatic):
# [bundled]     from src.orchestrator import run
    total_instructions = run("data/puzzles.csv", "submissions/sub_v1.csv")
"""
import argparse
import multiprocessing
import time
from typing import Optional
# [bundled] from src.utils import Puzzle, Instruction, load_puzzles, write_submission
# [bundled] from src.verifier import verify_program, quick_verify
# [bundled] from src.classifier import classify, PuzzleClass


# ---------------------------------------------------------------------------
# Per-puzzle solver dispatcher
# ---------------------------------------------------------------------------

def solve_puzzle(
    puzzle: Puzzle,
    *,
    sa_time: float = 2.0,
    n_starts: int = 3,
    verbose: bool = False,
) -> list[Instruction]:
    """
    Run the full solver portfolio for one puzzle. Returns the best valid program.

    Portfolio (in priority order):
      1. binary_shared  — pool-aware binary construction w/ prefix sharing (new primary)
      2. fallback       — simple binary construction (guaranteed valid ceiling)
      3. Specialised solvers based on classifier (tiny_bfs, factor_smooth, delta_cluster)
      4. Post-processing: dead-op removal + simulated annealing
    """
# [bundled]     from src.solvers import fallback, tiny_bfs, meet_in_middle, delta_cluster, factor_smooth
# [bundled]     from src.solvers import binary_shared
# [bundled]     from src.postpass import postprocess

    label = classify(puzzle)
    candidates: list[list[Instruction]] = []

    # --- Fallback (always — guaranteed valid) ---
    fb_prog = fallback.solve(puzzle)
    r = verify_program(puzzle, fb_prog)
    if r.valid:
        candidates.append(fb_prog)

    # --- binary_shared (new primary solver — always run) ---
    try:
        prog = binary_shared.solve(puzzle)
        r = verify_program(puzzle, prog)
        if r.valid:
            candidates.append(prog)
    except Exception:
        pass

    # --- Specialised solver based on label ---
    try:
        if label == PuzzleClass.TINY:
            prog = tiny_bfs.solve(puzzle)
        elif label == PuzzleClass.SMOOTH:
            prog = factor_smooth.solve(puzzle)
        elif label == PuzzleClass.CLUSTERED:
            prog = delta_cluster.solve(puzzle)
        else:
            prog = meet_in_middle.solve(puzzle)

        r = verify_program(puzzle, prog)
        if r.valid:
            candidates.append(prog)
    except Exception:
        pass

    # --- M-i-M multi-start (helps clustered/smooth puzzles) ---
    for seed in range(n_starts):
        try:
            prog = meet_in_middle.solve(puzzle, random_seed=seed)
            r = verify_program(puzzle, prog)
            if r.valid:
                candidates.append(prog)
        except Exception:
            pass

    if not candidates:
        raise RuntimeError(f"All solvers failed for puzzle {puzzle.id}")

    # Pick shortest so far
    best = min(candidates, key=len)

    # --- Post-processing ---
    try:
        improved = postprocess(puzzle, best, run_annealing=(sa_time > 0),
                               annealing_budget=sa_time)
        ok, count, err = quick_verify(puzzle, improved)
        if ok:
            best = improved
    except Exception:
        pass

    if verbose:
        print(f"  {puzzle.id} [{label.name}]: {len(best)} instructions")

    return best


# ---------------------------------------------------------------------------
# Worker function for multiprocessing
# ---------------------------------------------------------------------------

def _worker(args: tuple) -> tuple[str, list[list], int]:
    """
    Multiprocessing worker. Returns (puzzle_id, program_as_list_of_lists, count).
    """
    puzzle, sa_time, n_starts, verbose = args
    try:
        prog = solve_puzzle(puzzle, sa_time=sa_time, n_starts=n_starts, verbose=verbose)
        return puzzle.id, [inst.to_list() for inst in prog], len(prog)
    except Exception as e:
        # Emergency: use fallback
# [bundled]         from src.solvers import fallback
        prog = fallback.solve(puzzle)
        return puzzle.id, [inst.to_list() for inst in prog], len(prog)


# ---------------------------------------------------------------------------
# Main run function
# ---------------------------------------------------------------------------

def run(
    puzzles_path: str,
    out_path: str,
    *,
    cores: int = 1,
    sa_time: float = 2.0,
    n_starts: int = 3,
    verbose: bool = True,
) -> int:
    """
    Solve all puzzles and write submission CSV. Returns total instruction count.
    """
    puzzles = load_puzzles(puzzles_path)
    total_start = time.monotonic()

    if verbose:
        print(f"Loaded {len(puzzles)} puzzles. Solving with {cores} core(s)...")

    # Build arguments for workers
    worker_args = [(p, sa_time, n_starts, verbose and cores == 1) for p in puzzles]

    results: dict[str, list[Instruction]] = {}

    if cores > 1:
        with multiprocessing.Pool(processes=cores) as pool:
            for pid, prog_lists, count in pool.imap_unordered(_worker, worker_args, chunksize=1):
# [bundled]                 from src.utils import Instruction
                results[pid] = [Instruction(*row) for row in prog_lists]
                if verbose:
                    print(f"  {pid}: {count} instructions")
    else:
        for puzzle in puzzles:
            pid, prog_lists, count = _worker((puzzle, sa_time, n_starts, verbose))
# [bundled]             from src.utils import Instruction
            results[pid] = [Instruction(*row) for row in prog_lists]

    # Write submission
    write_submission(results, out_path)
    total = sum(len(v) for v in results.values())
    elapsed = time.monotonic() - total_start

    if verbose:
        print(f"\nTotal instructions: {total:,}")
        print(f"Elapsed: {elapsed:.1f}s")
        print(f"Submission written to: {out_path}")

    return total


# ---------------------------------------------------------------------------
# CLI entry point
# ---------------------------------------------------------------------------

def _cli():
    parser = argparse.ArgumentParser(description="Santa 2026 Elf Calculator Solver")
    parser.add_argument("--puzzles",  default="data/puzzles.csv", help="Path to puzzles.csv")
    parser.add_argument("--out",      default="submissions/submission.csv", help="Output CSV path")
    parser.add_argument("--cores",    type=int, default=1, help="Number of parallel workers")
    parser.add_argument("--sa-time",  type=float, default=2.0, help="SA time budget per puzzle (s)")
    parser.add_argument("--n-starts", type=int, default=3, help="M-i-M random restarts per puzzle")
    parser.add_argument("--verify",   action="store_true", help="Run full re-verification after solving")
    args = parser.parse_args()

    total = run(
        args.puzzles, args.out,
        cores=args.cores,
        sa_time=args.sa_time,
        n_starts=args.n_starts,
        verbose=True,
    )

    if args.verify:
# [bundled]         from src.verifier import replay_submission
        print("\nRunning full re-verification...")
        verified = replay_submission(args.puzzles, args.out)
        assert verified == total, f"Mismatch: {verified} vs {total}"
        print(f"Verification passed. Score: {verified:,}")


# [bundled-main] if __name__ == "__main__":
# [bundled-main]     _cli()


# ===========================================================================
# ENTRY POINT
# ===========================================================================
if __name__ == "__main__":
    _cli()


In [ ]:
import os
print('CPUs available:', os.cpu_count())
print('Input files:', os.listdir('/kaggle/input/santa-2026/'))
print('submit.py written:', os.path.exists('submit.py'))


In [ ]:
%%time
import subprocess, sys, os

PUZZLES  = '/kaggle/input/santa-2026/puzzles.csv'
OUT      = '/kaggle/working/submission.csv'
CORES    = max(1, (os.cpu_count() or 2) - 1)   # leave 1 CPU free
SA_TIME  = 30.0   # seconds SA budget per puzzle (increase for better score)
N_STARTS = 3      # MiM random restarts

print(f'Cores: {CORES}, SA budget: {SA_TIME}s/puzzle')

result = subprocess.run([
    sys.executable, 'submit.py',
    '--puzzles',  PUZZLES,
    '--out',      OUT,
    '--cores',    str(CORES),
    '--sa-time',  str(SA_TIME),
    '--n-starts', str(N_STARTS),
    '--verify',
], capture_output=False, text=True)

print('Return code:', result.returncode)
assert result.returncode == 0, 'Solver failed!'


In [ ]:
import pandas as pd
sub = pd.read_csv('/kaggle/working/submission.csv')
print('Rows:', len(sub))
print('Columns:', list(sub.columns))
print(sub.head(3))
# Count total instructions
total = sub['program'].apply(lambda x: len(__import__('json').loads(x))).sum()
print(f'Total instructions: {total:,}')
